In [ ]:
!pip install tensorflow tensorflow_datasets matplotlib tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.9/644.9 MB 586.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.3/106.3 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.5/224.5 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.7 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

dataset_name = "food101"

# Load only 30% of Food-101 train split (containing 75,000 images) (22,500 images)
full_dataset, dataset_info = tfds.load(
    dataset_name,
    split="train[:30%]",  # Take first 30% of the dataset
    as_supervised=True,
    with_info=True
)

num_classes = dataset_info.features["label"].num_classes
print(f"🍔 Loaded 30% of Food-101 (Total Images: 22,500)")

# Define train-validation split sizes
train_size = int(0.8 * 22500)  # 22,500 images
val_size = 22500 - train_size


# Split the dataset using 'take' and 'skip'
train_ds = full_dataset.take(train_size)
val_ds = full_dataset.skip(train_size)

print(f"✅ Train Images: {train_size}, Validation Images: {val_size}")

IMG_SIZE = 224
BATCH_SIZE = 16

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE)) / 255.0  # Normalize
    return image, label

# Apply preprocessing and batching to the datasets
train_ds = train_ds.map(preprocess).batch(BATCH_SIZE).shuffle(1000).prefetch(tf.data.AUTOTUNE) # Applying preprocess, batching, shuffling, and prefetching
val_ds = val_ds.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE) # Applying preprocess, batching, and prefetching

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...:   0%|          | 0/75750 [00:00<?, ? examples/s]

Shuffling /root/tensorflow_datasets/food101/incomplete.YISW99_2.0.0/food101-train.tfrecord*...:   0%|         …

Generating validation examples...:   0%|          | 0/25250 [00:00<?, ? examples/s]

Shuffling /root/tensorflow_datasets/food101/incomplete.YISW99_2.0.0/food101-validation.tfrecord*...:   0%|    …

Dataset food101 downloaded and prepared to /root/tensorflow_datasets/food101/2.0.0. Subsequent calls will reuse this data.
🍔 Loaded 30% of Food-101 (Total Images: 22,500)
✅ Train Images: 18000, Validation Images: 4500


In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze pretrained layers for fast training
base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.Conv2D(512, (3, 3), activation="relu", padding="same"),  # Additional convolutional layer
    tf.keras.layers.BatchNormalization(),  # Normalize activations
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(num_classes, activation="softmax")  # Output layer
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 7, 7, 512)           │       5,898,752 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 7, 7, 512)           │           2,048 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 512)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 101)                 │          51,813 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 8,210,597 (31.32 MB)

 Trainable params: 5,951,589 (22.70 MB)

 Non-trainable params: 2,259,008 (8.62 MB)

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report

# Custom Callback for tqdm Progress Bar
class TQDMProgressBar(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.epochs = self.params['epochs']

    def on_epoch_begin(self, epoch, logs=None):
        self.progress_bar = tqdm(total=self.params['steps'],
                                 desc=f"Epoch {epoch + 1}/{self.epochs}",
                                 leave=True, dynamic_ncols=True)

    def on_batch_end(self, batch, logs=None):
        self.progress_bar.update(1)

    def on_epoch_end(self, epoch, logs=None):
        self.progress_bar.close()
        print(f"✅ Epoch {epoch + 1}/{self.epochs} - "
              f"Loss: {logs['loss']:.4f}, Accuracy: {logs['accuracy']:.4f}, "
              f"Val Loss: {logs['val_loss']:.4f}, Val Accuracy: {logs['val_accuracy']:.4f}")

# Train on GPU
with tf.device("/GPU:0"):
    history = model.fit(train_ds, validation_data=val_ds, epochs=5, callbacks=[TQDMProgressBar()])

# Extract training history
train_loss = history.history['loss']
val_loss = history.history['val_loss']
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

# Plot Training & Validation Loss/Accuracy
plt.figure(figsize=(12, 5))

# Loss Graph
plt.subplot(1, 2, 1)
plt.plot(train_loss, label="Train Loss")
plt.plot(val_loss, label="Val Loss", linestyle="dashed")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss over Epochs")

# Accuracy Graph
plt.subplot(1, 2, 2)
plt.plot(train_acc, label="Train Accuracy")
plt.plot(val_acc, label="Val Accuracy", linestyle="dashed")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy over Epochs")

plt.show()

# Compute F1 Score
y_true, y_pred = [], []

for images, labels in val_ds:
    preds = model.predict(images)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

from sklearn.metrics import f1_score

f1 = f1_score(y_true, y_pred, average="weighted")
print(f"📊 Weighted F1-Score on Validation Set: {f1:.4f}")

# Show Classification Report
print(classification_report(y_true, y_pred, digits=4))


Epoch 1/5:   0%|          | 0/1125 [00:00<?, ?it/s]

Epoch 1/5
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.3308 - loss: 2.8855✅ Epoch 1/5 - Loss: 2.4632, Accuracy: 0.4027, Val Loss: 2.0853, Val Accuracy: 0.4821
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 2222s 2s/step - accuracy: 0.3309 - loss: 2.8851 - val_accuracy: 0.4821 - val_loss: 2.0853


Epoch 2/5:   0%|          | 0/1125 [00:00<?, ?it/s]

Epoch 2/5
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6474 - loss: 1.4118✅ Epoch 2/5 - Loss: 1.4715, Accuracy: 0.6293, Val Loss: 2.1252, Val Accuracy: 0.4789
1125/1125 ━━━━━━━━━━━━━━━━━━━━ 2184s 2s/step - accuracy: 0.6474 - loss: 1.4119 - val_accuracy: 0.4789 - val_loss: 2.1252


Epoch 3/5:   0%|          | 0/1125 [00:00<?, ?it/s]

Epoch 3/5
  46/1125 ━━━━━━━━━━━━━━━━━━━━ 28:32 2s/step - accuracy: 0.8137 - loss: 0.8072

In [ ]:
# test random 3 images on this
import random

# Load test dataset
test_ds = tfds.load(dataset_name, split="validation", as_supervised=True)

# Select 3 random images
test_images = []
test_labels = []

for image, label in test_ds.take(3):
    test_images.append(image)
    test_labels.append(label)

# Preprocess images
test_images = [preprocess(img, lbl)[0] for img, lbl in zip(test_images, test_labels)]
test_images = np.array(test_images)

# Get predictions
predictions = model.predict(test_images)
predicted_labels = np.argmax(predictions, axis=1)

# Display results
plt.figure(figsize=(10, 5))
for i in range(3):
    plt.subplot(1, 3, i + 1)
    plt.imshow(test_images[i])
    plt.axis("off")
    plt.title(f"Pred: {dataset_info.features['label'].int2str(predicted_labels[i])}")

plt.show()
